# 🔗 Day 01: Linked List Fundamentals — Thinking in Pointers

---

## Why Linked Lists Matter

Arrays store elements in a contiguous block of memory. That's great for random access (`arr[5]` is instant), but terrible for insertions in the middle — you have to shift everything.

A **linked list** is different. Each element (node) stores its value AND a reference to the next node. Nodes can live anywhere in memory — they're connected by pointers, not by position.

```
Array:       [1][2][3][4][5]  ← all together in memory

Linked List: [1]→[2]→[3]→[4]→[5]→None  ← scattered, connected by arrows
```

**Why do interviews love them?** Because they test whether you can think about **references** — who points to whom, what happens when you rearrange pointers. It's the foundation for trees, graphs, and everything else.

---

### Array vs Linked List — Quick Comparison

```
╔══════════════════════════════════════════════════════════════╗
║  Operation         │  Array      │  Linked List            ║
╠══════════════════════════════════════════════════════════════╣
║  Access by index   │  O(1)       │  O(n) — must traverse   ║
║  Insert at head    │  O(n)       │  O(1) — just repoint    ║
║  Insert at tail    │  O(1)*      │  O(n) or O(1) with tail ║
║  Insert in middle  │  O(n)       │  O(1) if you have ref   ║
║  Delete            │  O(n)       │  O(1) if you have ref   ║
║  Memory            │  Compact    │  Extra space for ptrs   ║
╚══════════════════════════════════════════════════════════════╝
```

---

## 🏗️ Building a Linked List from Scratch

The fundamental building block is a **Node**: it holds a value and a pointer to the next node.

Every LeetCode linked list problem gives you this definition (don't memorize it, understand it):

In [ ]:
class ListNode:
    """The building block of a singly linked list."""
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next  # Points to next node (or None if end)

# Creating a list: 1 → 2 → 3 → None
head = ListNode(1)
head.next = ListNode(2)
head.next.next = ListNode(3)

# Helper: build a linked list from a Python list (you'll use this a LOT for testing)
def build_list(values):
    """[1,2,3] → 1→2→3→None"""
    dummy = ListNode(0)
    curr = dummy
    for v in values:
        curr.next = ListNode(v)
        curr = curr.next
    return dummy.next

# Helper: print a linked list
def print_list(head):
    """1→2→3→None"""
    parts = []
    while head:
        parts.append(str(head.val))
        head = head.next
    print(" → ".join(parts) + " → None")

# Test it
my_list = build_list([1, 2, 3, 4, 5])
print_list(my_list)

---

## 🧠 The Dummy Head Trick

This is probably the most important linked list technique. Before we solve any problem, let me explain it.

**The problem:** Many operations might change the head of the list. For example, if you're removing elements and the first element needs to be removed — now the head changes. This leads to ugly special-case code.

**The solution:** Create a fake "dummy" node that points to the real head. Work with `dummy.next` as the starting point. At the end, return `dummy.next` (which is the real head, possibly different from the original).

```
Before: head → [1] → [2] → [3] → None
                      ↑ What if we need to remove head?

With dummy:
         dummy → [1] → [2] → [3] → None
         ↑ always safe to reference dummy.next as head
```

**Rule of thumb:** If the head might change, use a dummy node. When in doubt, use one anyway — it won't hurt.

In [ ]:
# Dummy head in action: Remove all nodes with a given value

def removeElements(head, val):
    """
    LC #203 — Remove all nodes with value = val.
    Time: O(n), Space: O(1)
    """
    dummy = ListNode(0)
    dummy.next = head
    
    prev = dummy
    curr = head
    
    while curr:
        if curr.val == val:
            prev.next = curr.next  # Skip this node
        else:
            prev = curr            # Move prev forward
        curr = curr.next
    
    return dummy.next  # New head (might be different!)

# Test: Remove all 2s from [1,2,2,3,4,2,5]
lst = build_list([1, 2, 2, 3, 4, 2, 5])
print("Before:", end=" "); print_list(lst)
result = removeElements(lst, 2)
print("After: ", end=" "); print_list(result)

# Edge case: remove head itself
lst2 = build_list([2, 2, 3])
result2 = removeElements(lst2, 2)
print("Remove head:", end=" "); print_list(result2)

---

## 🎯 Problem 1: Reverse Linked List (LC #206)

The **absolute most important** linked list problem. It's the building block for at least 10 other problems. If you can't reverse a linked list in your sleep, nothing else will make sense.

```
Input:  1 → 2 → 3 → 4 → 5 → None
Output: 5 → 4 → 3 → 2 → 1 → None
```

**The visual:** Imagine you're walking along the list, and at each node, you flip the arrow to point backward instead of forward.

```
Step 0:  1 → 2 → 3 → None     prev = None, curr = 1
Step 1:  1 ← 2 → 3 → None     prev = 1, curr = 2  (flipped 1's arrow)
Step 2:  1 ← 2 ← 3 → None     prev = 2, curr = 3  (flipped 2's arrow)
Step 3:  1 ← 2 ← 3             prev = 3, curr = None (flipped 3's arrow)
                                 ↑ new head!
```

**Three pointers:** `prev` (starts at None), `curr` (starts at head), `next_temp` (saves the next node before we flip).

In [ ]:
# Iterative — this is the one you should know cold
def reverseList(head):
    """
    Reverse a singly linked list.
    Time: O(n), Space: O(1)
    """
    prev = None
    curr = head
    
    while curr:
        next_temp = curr.next  # Save next before we break the link
        curr.next = prev       # Flip the arrow
        prev = curr            # Move prev forward
        curr = next_temp       # Move curr forward
    
    return prev  # prev is now the new head

lst = build_list([1, 2, 3, 4, 5])
print("Before:", end=" "); print_list(lst)
reversed_lst = reverseList(lst)
print("After: ", end=" "); print_list(reversed_lst)

In [ ]:
# Recursive version — elegant but uses O(n) stack space
def reverseList_recursive(head):
    """
    Recursive reversal.
    Time: O(n), Space: O(n) call stack
    
    Think of it this way:
    "Reverse everything after me, then make the next node point back to me."
    """
    # Base case: empty list or single node
    if not head or not head.next:
        return head
    
    # Recursively reverse the rest
    new_head = reverseList_recursive(head.next)
    
    # Make the next node point back to me
    head.next.next = head
    head.next = None  # I'm now the tail
    
    return new_head

lst = build_list([1, 2, 3, 4, 5])
reversed_lst = reverseList_recursive(lst)
print("Recursive:", end=" "); print_list(reversed_lst)

---

## 🎯 Problem 2: Merge Two Sorted Lists (LC #21)

Another fundamental one. It's the merge step of merge sort, but for linked lists.

```
Input:  1 → 2 → 4
        1 → 3 → 4
Output: 1 → 1 → 2 → 3 → 4 → 4
```

**The approach:** Use a dummy head. Compare the fronts of both lists, take the smaller one, advance that pointer. Repeat.

In [ ]:
def mergeTwoLists(list1, list2):
    """
    Merge two sorted lists into one sorted list.
    Time: O(n + m), Space: O(1) — we're reusing existing nodes
    """
    dummy = ListNode(0)  # Dummy head — don't have to worry about which list starts
    tail = dummy
    
    while list1 and list2:
        if list1.val <= list2.val:
            tail.next = list1
            list1 = list1.next
        else:
            tail.next = list2
            list2 = list2.next
        tail = tail.next
    
    # Attach whichever list has remaining nodes
    tail.next = list1 or list2
    
    return dummy.next

l1 = build_list([1, 2, 4])
l2 = build_list([1, 3, 4])
merged = mergeTwoLists(l1, l2)
print("Merged:", end=" "); print_list(merged)

---

## 🎯 Problem 3: Linked List Cycle (LC #141)

Can you detect if a linked list has a cycle, using O(1) space?

This is where **Floyd's Tortoise and Hare** algorithm enters. Two pointers:
- **Slow** moves 1 step at a time
- **Fast** moves 2 steps at a time

If there's a cycle, fast will eventually lap slow — they'll meet. If there's no cycle, fast reaches the end.

Think of it like two runners on a circular track. The faster one will always catch up to the slower one.

In [ ]:
def hasCycle(head):
    """
    Floyd's cycle detection.
    Time: O(n), Space: O(1)
    """
    slow = fast = head
    
    while fast and fast.next:
        slow = slow.next       # 1 step
        fast = fast.next.next  # 2 steps
        
        if slow == fast:  # They met! Cycle exists.
            return True
    
    return False  # Fast reached end → no cycle

# Test with cycle
a = ListNode(1)
b = ListNode(2)
c = ListNode(3)
a.next = b; b.next = c; c.next = b  # cycle: 3 → 2 → 3 → 2...
print(f"Has cycle: {hasCycle(a)}")  # True

# Test without cycle
no_cycle = build_list([1, 2, 3])
print(f"Has cycle: {hasCycle(no_cycle)}")  # False

---

## 🗺️ Linked List Fundamentals Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  LINKED LIST CORE TECHNIQUES                                        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. DUMMY HEAD: Create fake node before head to simplify edge cases ║
║     → Use when head might change (remove, merge, etc.)              ║
║                                                                      ║
║  2. REVERSAL: prev/curr/next_temp — flip arrows one by one          ║
║     → iterative O(1) space, recursive O(n) space                    ║
║                                                                      ║
║  3. MERGE: dummy + tail pointer, compare fronts, attach smaller     ║
║     → Same pattern for merge sort, merge K lists                    ║
║                                                                      ║
║  4. FLOYD'S: slow (1 step) + fast (2 steps)                        ║
║     → Cycle detection, find middle, find cycle start                ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 206 | Reverse Linked List | Easy | Reversal |
| 21 | Merge Two Sorted Lists | Easy | Merge + dummy |
| 141 | Linked List Cycle | Easy | Floyd's |
| 203 | Remove Linked List Elements | Easy | Dummy head |
| 707 | Design Linked List | Medium | All basics |
| 237 | Delete Node in a Linked List | Medium | Trick question |
| 83 | Remove Duplicates from Sorted List | Easy | Pointer skip |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Linked list = nodes connected by references, NOT by position   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Dummy head eliminates head-change edge cases — use it freely   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Reversal uses 3 pointers: prev, curr, next_temp — know this   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Floyd's (slow/fast) detects cycles in O(1) space              │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Always draw it out! Pointer problems are visual problems       │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Linked List Two-Pointer Techniques

Finding middles, removing from the end, and the cycle start trick.

---

### 🎉 Linked List Foundations Built!
Reverse, merge, detect cycles — the three pillars of linked list mastery.